# 05 Fill Additive Marking — local_3x3 adaptive bottleneck CNN

Version 20 is copied conceptually from `05-fill-additive-marking-local-3x3-19-gpt.ipynb`.

It keeps the same fast unique-patch training method, but replaces fixed H=4 with an adaptive hidden-channel sweep:

```text
HIDDEN_CANDIDATES = [4, 6, 8, 12]
```

For each task, the notebook tries the smallest model first and saves the first exported ONNX model that is exact on visible examples.

The submitted architecture is still:

```text
Input 1×10×30×30
Conv1 3×3, 10→H, padding=1
ReLU
Conv2 1×1, H→10
Output 1×10×30×30
```

Safety gates remain unchanged: all 8 tasks must save models, and exported ONNX validation must have `wrong = 0` for every task.

Export uses `dynamo=False` to avoid requiring `onnxscript` on Kaggle.

In [ ]:
# Common helpers: paths, data loading, ONNX runtime validation, model reporting.

import json, math, shutil, subprocess, sys, time, zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

BATCH, CH, GRID_H, GRID_W = 1, 10, 30, 30
PATCH_DIM = CH * 3 * 3
FAMILY = 'fill_enclosed_regions'
MODEL_VERSION = 'fill-additive-local3x3-adaptive-bottleneck-v0.20'
HIDDEN_CANDIDATES = [4, 6, 8, 12]

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        __import__(import_name)
        return
    except Exception:
        print(f'Installing missing package: {pip_name}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

ensure_package('onnx')
ensure_package('onnxruntime')
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort

def default_paths():
    kaggle_dir = Path('/kaggle/input/competitions/neurogolf-2026')
    if kaggle_dir.exists():
        root, data_dir = Path('/kaggle/working'), kaggle_dir
    else:
        root = Path.cwd()
        data_dir = root / 'competition_material' / 'taskfiles'
        if not data_dir.exists():
            data_dir = root / 'competition_material'
    out_dir = root / 'working_submission' / FAMILY
    out_dir.mkdir(parents=True, exist_ok=True)
    return data_dir, out_dir

def load_task_type_map(path='/kaggle/input/datasets/prince22466/task-type-map-csv/task_type_map.csv'):
    candidates = [Path(path), Path('task_groups/task_type_map.csv'), Path('Co_Kaggle/g3/task_groups/task_type_map.csv')]
    for p in candidates:
        if p.exists():
            return pd.read_csv(p, dtype={'task_id': str})
    raise FileNotFoundError(f'task_type_map.csv not found in: {candidates}')

def task_path(data_dir, task_id):
    name = f'{task_id}.json' if str(task_id).startswith('task') else f'task{int(task_id):03d}.json'
    for p in [Path(data_dir) / name, Path(data_dir) / 'taskfiles' / name]:
        if p.exists():
            return p
    raise FileNotFoundError(name)

def load_task(data_dir, task_id):
    with task_path(data_dir, task_id).open('r', encoding='utf-8') as f:
        return json.load(f)

def all_examples(task):
    return task.get('train', []) + task.get('test', []) + task.get('arc-gen', [])

def grid_to_chw(grid):
    arr = np.zeros((CH, GRID_H, GRID_W), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if 0 <= r < GRID_H and 0 <= c < GRID_W and 0 <= int(color) < CH:
                arr[int(color), r, c] = 1.0
    return arr

def grid_to_tensor(grid):
    return grid_to_chw(grid)[None].astype(np.float32)

def run_onnx(path, input_grid):
    sess = ort.InferenceSession(str(path), providers=['CPUExecutionProvider'])
    out = sess.run(['output'], {'input': grid_to_tensor(input_grid)})[0]
    return (out > 0).astype(np.float32)

def visible_validation_summary(model_path, task):
    right = wrong = 0
    for ex in all_examples(task):
        expected = grid_to_tensor(ex['output'])
        actual = run_onnx(model_path, ex['input'])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
    return {'right': right, 'wrong': wrong, 'total': right + wrong, 'accuracy': right / (right + wrong) if right + wrong else None}

def split_validation_summary(model_path, task):
    rows = {}
    for split, examples in {'train': task.get('train', []), 'test': task.get('test', []), 'arc_gen': task.get('arc-gen', [])}.items():
        right = wrong = 0
        for ex in examples:
            expected = grid_to_tensor(ex['output'])
            actual = run_onnx(model_path, ex['input'])
            if np.array_equal(actual, expected):
                right += 1
            else:
                wrong += 1
        total = right + wrong
        rows[split] = {'right': right, 'wrong': wrong, 'total': total, 'accuracy': right / total if total else None}
    rows['visible_all'] = visible_validation_summary(model_path, task)
    return rows

def count_model_params(model_path):
    model = onnx.load(str(model_path))
    return int(sum(math.prod(init.dims) if init.dims else 1 for init in model.graph.initializer))

def static_memory_from_shapes(model_path):
    model = onnx.shape_inference.infer_shapes(onnx.load(str(model_path)))
    total = 0
    for value in list(model.graph.value_info):
        dims = []
        for dim in value.type.tensor_type.shape.dim:
            if not dim.HasField('dim_value') or dim.dim_value <= 0:
                dims = []
                break
            dims.append(dim.dim_value)
        if dims:
            total += math.prod(dims) * 4
    return int(total)

def model_report(model_path, task):
    model = onnx.load(str(model_path))
    ops = Counter(node.op_type for node in model.graph.node)
    params = count_model_params(model_path)
    static_mem = static_memory_from_shapes(model_path)
    perf = split_validation_summary(model_path, task)
    return {
        'architecture': {'nodes': len(model.graph.node), 'op_counts': dict(ops), 'params': params, 'file_size_bytes': Path(model_path).stat().st_size},
        'memory_profile': {'static_memory_bytes': static_mem, 'runtime_memory_bytes': static_mem, 'params': params},
        'performance': perf,
    }

def create_submission_zip(model_dir, zip_path=None):
    model_dir = Path(model_dir)
    zip_path = model_dir / 'submission.zip' if zip_path is None else Path(zip_path)
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(model_dir.glob('task*.onnx')):
            zf.write(p, p.name)
    return zip_path

DATA_DIR, OUT_DIR = default_paths()
print('DATA_DIR =', DATA_DIR)
print('OUT_DIR =', OUT_DIR)
print('MODEL_VERSION =', MODEL_VERSION)
print('HIDDEN_CANDIDATES =', HIDDEN_CANDIDATES)
print('torch/onnx/ort =', torch.__version__, onnx.__version__, ort.__version__)

In [ ]:
# Select only the 8 local_3x3 fill/additive marking tasks.

task_map = load_task_type_map()
family_df = task_map[task_map.primary_family == FAMILY].copy()
local_3x3_df = family_df[family_df.candidate_flags.str.contains('local_3x3_consistent', na=False)].copy()
task_ids = local_3x3_df['task_id'].tolist()

print('family:', FAMILY)
print('family tasks:', len(family_df))
print('selected local_3x3 tasks:', len(task_ids))
display(local_3x3_df[['task_id', 'n_train', 'n_test', 'n_arc_gen', 'shape_relation', 'local_3x3_score', 'local_3x3_conflicts', 'candidate_flags']])

assert len(task_ids) == 8, f'Expected 8 local_3x3 tasks, got {len(task_ids)}'
assert (local_3x3_df['local_3x3_score'].astype(float) == 1.0).all()
assert (local_3x3_df['local_3x3_conflicts'].astype(int) == 0).all()

In [ ]:
# Adaptive H fast unique-patch training/export.

def target_vec_at(output_grid, r, c):
    y = np.zeros((CH,), dtype=np.float32)
    if r < len(output_grid) and len(output_grid) > 0 and c < len(output_grid[0]):
        color = int(output_grid[r][c])
        if 0 <= color < CH:
            y[color] = 1.0
    return y

def patch_at_chw(x, r, c):
    patch = np.zeros((CH, 3, 3), dtype=np.float32)
    for dy in range(-1, 2):
        rr = r + dy
        if rr < 0 or rr >= GRID_H:
            continue
        for dx in range(-1, 2):
            cc = c + dx
            if 0 <= cc < GRID_W:
                patch[:, dy + 1, dx + 1] = x[:, rr, cc]
    return patch

def build_unique_patch_dataset(task):
    mapping, conflicts, total = {}, [], 0
    for ex_i, ex in enumerate(all_examples(task)):
        x = grid_to_chw(ex['input'])
        for r in range(GRID_H):
            for c in range(GRID_W):
                patch = patch_at_chw(x, r, c)
                y = target_vec_at(ex['output'], r, c)
                key = patch.tobytes()
                total += 1
                if key not in mapping:
                    mapping[key] = (patch, y)
                elif mapping[key][1].tobytes() != y.tobytes():
                    conflicts.append((ex_i, r, c))
    xs = np.stack([v[0] for v in mapping.values()]).astype(np.float32)
    ys = np.stack([v[1] for v in mapping.values()]).astype(np.float32)
    return xs, ys, {'unique_patches': len(xs), 'total_positions': total, 'conflict_count': len(conflicts), 'conflicts': conflicts[:10]}

class PatchMLP(nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.fc1 = nn.Linear(PATCH_DIM, hidden_channels)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_channels, CH)
    def forward(self, patch):
        return self.fc2(self.relu(self.fc1(patch.reshape(patch.shape[0], -1))))

class BottleneckLocal3x3(nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(CH, hidden_channels, kernel_size=3, padding=1, bias=True)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(hidden_channels, CH, kernel_size=1, padding=0, bias=True)
    def forward(self, x):
        return self.conv2(self.relu(self.conv1(x)))

def copy_mlp_to_conv(mlp, hidden_channels):
    conv = BottleneckLocal3x3(hidden_channels=hidden_channels)
    with torch.no_grad():
        conv.conv1.weight.copy_(mlp.fc1.weight.reshape(hidden_channels, CH, 3, 3))
        conv.conv1.bias.copy_(mlp.fc1.bias)
        conv.conv2.weight.copy_(mlp.fc2.weight.reshape(CH, hidden_channels, 1, 1))
        conv.conv2.bias.copy_(mlp.fc2.bias)
    conv.eval()
    return conv

def exact_patch_stats(model, x, y):
    with torch.no_grad():
        pred = (model(x) > 0).to(y.dtype)
        ok = (pred == y).all(dim=1)
    right = int(ok.sum().item())
    total = int(y.shape[0])
    return right, total - right, total

def train_hidden_for_task(task, *, task_id, hidden_channels, max_epochs, seeds, lrs=(0.05, 0.02), check_every=20, positive_weight=3.0):
    x_np, y_np, dataset_info = build_unique_patch_dataset(task)
    if dataset_info['conflict_count']:
        return None, {'ok': False, 'task_id': task_id, 'hidden_channels': hidden_channels, 'reason': 'local patch conflicts', **dataset_info}

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    x = torch.from_numpy(x_np).to(device)
    y = torch.from_numpy(y_np).to(device)
    pos_weight_tensor = torch.full((CH,), float(positive_weight), device=device)

    attempts, best_state, best_wrong, best_loss = [], None, 10**9, float('inf')
    for lr in lrs:
        for seed in seeds:
            torch.manual_seed(seed); np.random.seed(seed)
            model = PatchMLP(hidden_channels=hidden_channels).to(device)
            optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
            seed_best = {'hidden_channels': hidden_channels, 'lr': lr, 'seed': seed, 'epoch': None, 'wrong': 10**9, 'loss': float('inf')}
            start = time.time()

            for epoch in range(1, max_epochs + 1):
                optimizer.zero_grad(set_to_none=True)
                logits = model(x)
                bce = criterion(logits, y)
                margin = (torch.relu(1.5 - logits) * y + torch.relu(1.5 + logits) * (1.0 - y)).mean()
                loss = bce + 0.10 * margin
                loss.backward(); optimizer.step()

                if epoch == 1 or epoch % check_every == 0 or epoch == max_epochs:
                    right, wrong, total = exact_patch_stats(model, x, y)
                    loss_value = float(loss.item())
                    if wrong < seed_best['wrong'] or (wrong == seed_best['wrong'] and loss_value < seed_best['loss']):
                        seed_best = {'hidden_channels': hidden_channels, 'lr': lr, 'seed': seed, 'epoch': epoch, 'right': right, 'wrong': wrong, 'total': total, 'loss': loss_value, 'elapsed_sec': time.time() - start}
                    if wrong < best_wrong or (wrong == best_wrong and loss_value < best_loss):
                        best_wrong, best_loss = wrong, loss_value
                        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                    if wrong == 0:
                        attempts.append(seed_best)
                        model_cpu = PatchMLP(hidden_channels=hidden_channels)
                        model_cpu.load_state_dict({k: v.cpu() for k, v in model.state_dict().items()})
                        model_cpu.eval()
                        conv_model = copy_mlp_to_conv(model_cpu, hidden_channels=hidden_channels)
                        return conv_model, {'ok': True, 'task_id': task_id, 'trainer': 'adaptive_unique_patch_cnn', 'model_version': MODEL_VERSION, 'hidden_channels': hidden_channels, 'lr': lr, 'seed': seed, 'epoch': epoch, 'visible_patch_right': right, 'visible_patch_wrong': wrong, 'visible_patch_total': total, 'loss': loss_value, 'attempts': attempts, **dataset_info}

            attempts.append(seed_best)
            print(f"{task_id} H={hidden_channels} lr={lr} seed={seed}: unique={dataset_info['unique_patches']} best_wrong={seed_best['wrong']} epoch={seed_best['epoch']} elapsed={seed_best.get('elapsed_sec', 0):.1f}s")

    best_conv = None
    if best_state is not None:
        best_mlp = PatchMLP(hidden_channels=hidden_channels)
        best_mlp.load_state_dict(best_state)
        best_mlp.eval()
        best_conv = copy_mlp_to_conv(best_mlp, hidden_channels=hidden_channels)
    return best_conv, {'ok': False, 'task_id': task_id, 'trainer': 'adaptive_unique_patch_cnn', 'model_version': MODEL_VERSION, 'hidden_channels': hidden_channels, 'reason': f'H={hidden_channels} did not reach exact unique-patch match', 'best_visible_patch_wrong': best_wrong, 'best_loss': best_loss, 'attempts': attempts, **dataset_info}

def hidden_plan_for_task(task_id):
    # Keep H=4 cheap for easy tasks; increase only if exact fit fails.
    return [
        {'hidden_channels': 4, 'max_epochs': 1200, 'seeds': (0, 1, 2, 3), 'lrs': (0.05, 0.02)},
        {'hidden_channels': 6, 'max_epochs': 1800, 'seeds': (0, 1, 2, 3, 4, 5), 'lrs': (0.05, 0.02)},
        {'hidden_channels': 8, 'max_epochs': 2200, 'seeds': (0, 1, 2, 3, 4, 5), 'lrs': (0.05, 0.02)},
        {'hidden_channels': 12, 'max_epochs': 2200, 'seeds': (0, 1, 2, 3, 4, 5), 'lrs': (0.05, 0.02)},
    ]

def train_adaptive_for_task(task, task_id):
    all_attempts = []
    best_failure = None
    for cfg in hidden_plan_for_task(task_id):
        H = cfg['hidden_channels']
        print(f'-- trying {task_id} with H={H}')
        model, info = train_hidden_for_task(task, task_id=task_id, **cfg)
        all_attempts.append({'hidden_channels': H, 'ok': info.get('ok'), 'reason': info.get('reason'), 'best_visible_patch_wrong': info.get('best_visible_patch_wrong'), 'attempts': info.get('attempts')})
        if info.get('ok'):
            info['adaptive_attempts'] = all_attempts
            return model, info
        if best_failure is None or info.get('best_visible_patch_wrong', 10**9) < best_failure.get('best_visible_patch_wrong', 10**9):
            best_failure = info

    best_failure = best_failure or {'task_id': task_id}
    best_failure.update({'ok': False, 'reason': 'no hidden candidate reached exact unique-patch match', 'adaptive_attempts': all_attempts})
    return None, best_failure

def export_torch_model_to_onnx(model, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    model.eval()
    dummy = torch.zeros((BATCH, CH, GRID_H, GRID_W), dtype=torch.float32)
    with torch.no_grad():
        torch.onnx.export(model, dummy, str(path), input_names=['input'], output_names=['output'], opset_version=10, do_constant_folding=True, dynamo=False)
    onnx.checker.check_model(str(path))
    return path

In [ ]:
# Build one adaptive bottleneck ONNX model per selected local_3x3 task.

for old_model_path in OUT_DIR.glob('task*.onnx'):
    old_model_path.unlink()

rows = []
for task_id in task_ids:
    print('\n' + '=' * 80)
    print('training', task_id)
    task = load_task(DATA_DIR, task_id)
    model, info = train_adaptive_for_task(task, task_id)
    row = {'task_id': task_id, **info}

    if model is not None and info.get('ok'):
        path = OUT_DIR / f'{task_id}.onnx'
        export_torch_model_to_onnx(model, path)
        summary = visible_validation_summary(path, task)
        row.update({'saved': summary['wrong'] == 0, 'path': str(path), 'onnx_visible_right': summary['right'], 'onnx_visible_wrong': summary['wrong'], 'onnx_visible_total': summary['total']})
        if summary['wrong'] != 0:
            path.unlink(missing_ok=True)
            row['saved'] = False
            row['reason'] = 'unique-patch fit exact, but exported ONNX failed full-grid validation'
    else:
        row['saved'] = False

    rows.append(row)
    print({k: row.get(k) for k in ['task_id', 'saved', 'hidden_channels', 'unique_patches', 'visible_patch_wrong', 'onnx_visible_wrong', 'reason']})

result_df = pd.DataFrame(rows)
display(result_df)
saved_count = int(result_df['saved'].sum()) if len(result_df) else 0
print('selected local_3x3 tasks:', len(task_ids))
print('models saved:', saved_count)
print('hidden channel usage:', result_df[result_df['saved']].groupby('hidden_channels').size().to_dict() if saved_count else {})
assert saved_count == len(task_ids), f'adaptive bottleneck only saved {saved_count}/{len(task_ids)} models; not safe to submit.'
assert (result_df['onnx_visible_wrong'].fillna(1).astype(int) == 0).all(), 'At least one saved ONNX model is not exact.'

In [ ]:
# Validate saved ONNX models again on visible examples, split by task.

validate_rows = []
for row in rows:
    if not row.get('saved'):
        continue
    task = load_task(DATA_DIR, row['task_id'])
    summary = visible_validation_summary(row['path'], task)
    validate_rows.append({'task_id': row['task_id'], 'hidden_channels': row.get('hidden_channels'), 'right': summary['right'], 'wrong': summary['wrong'], 'total': summary['total']})
validate_df = pd.DataFrame(validate_rows)
display(validate_df)
assert len(validate_df) == len(task_ids)
assert (validate_df['wrong'].astype(int) == 0).all()

In [ ]:
# Architecture, memory, and estimated competition-cost report.

report_rows = []
for row in rows:
    if not row.get('saved'):
        continue
    task = load_task(DATA_DIR, row['task_id'])
    report = model_report(row['path'], task=task)
    arch, mem, perf = report['architecture'], report['memory_profile'], report['performance']
    report_rows.append({
        'task_id': row['task_id'],
        'model_version': MODEL_VERSION,
        'hidden_channels': row.get('hidden_channels'),
        'unique_patches': row.get('unique_patches'),
        'file_size_bytes': arch.get('file_size_bytes'),
        'params': arch.get('params'),
        'nodes': arch.get('nodes'),
        'op_counts': json.dumps(arch.get('op_counts', {}), sort_keys=True),
        'static_memory_bytes': mem.get('static_memory_bytes'),
        'runtime_memory_bytes': mem.get('runtime_memory_bytes'),
        'estimated_cost_static': arch.get('params') + mem.get('static_memory_bytes'),
        'estimated_cost_runtime': arch.get('params') + mem.get('runtime_memory_bytes'),
        'train_right': perf['train']['right'],
        'train_total': perf['train']['total'],
        'train_accuracy': perf['train']['accuracy'],
        'test_right': perf['test']['right'],
        'test_total': perf['test']['total'],
        'test_accuracy': perf['test']['accuracy'],
        'arc_gen_right': perf['arc_gen']['right'],
        'arc_gen_total': perf['arc_gen']['total'],
        'arc_gen_accuracy': perf['arc_gen']['accuracy'],
        'visible_right': perf['visible_all']['right'],
        'visible_total': perf['visible_all']['total'],
        'visible_accuracy': perf['visible_all']['accuracy'],
    })
profile_df = pd.DataFrame(report_rows)
display(profile_df)
display(profile_df[['hidden_channels', 'params', 'static_memory_bytes', 'runtime_memory_bytes', 'estimated_cost_static', 'estimated_cost_runtime']])
display(profile_df.groupby('hidden_channels')[['params', 'estimated_cost_static', 'estimated_cost_runtime']].mean())
assert len(profile_df) == len(task_ids)
assert (profile_df['visible_accuracy'] == 1.0).all()

In [ ]:
# Create submission.zip only after all gates pass.

zip_path = create_submission_zip(OUT_DIR)
submission_zip = Path('/kaggle/working/submission.zip') if Path('/kaggle/working').exists() else Path.cwd() / 'submission.zip'
shutil.copy2(zip_path, submission_zip)
manifest = {
    'family': FAMILY,
    'model_version': MODEL_VERSION,
    'hidden_candidates': HIDDEN_CANDIDATES,
    'task_count': len(task_ids),
    'saved_count': saved_count,
    'hidden_channel_usage': result_df[result_df['saved']].groupby('hidden_channels').size().to_dict(),
    'out_dir': str(OUT_DIR),
    'submission_zip': str(submission_zip),
}
manifest_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)
profile_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_profile.csv'
profile_df.to_csv(profile_path, index=False)
print('family zip:', zip_path)
print('kaggle submission zip:', submission_zip)
print('manifest:', manifest_path)
print('profile:', profile_path)
manifest